## Базовая функциональность Часть 1

## Переиндексация

Метод reindex - создание нового объекта, данные в котором согласуются с новым индексом

In [1]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

obj = Series([4.5, 7.2, -5.3, 3.6], index=['d', 'b', 'a', 'c'])
obj

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

Если вызвать reindex для этого объекта Series, то данные будут реорганизованы в соответствии с новым индексом, а если
каких-то из имеющихся в этом индексе значений раньше не было, то вместо них будут подставлены отсутствующие
значения:

In [2]:
arr = np.array([1, 2, 4, 5, 100])
arr_masked = np.ma.masked_equal(arr, 100)
df = pd.DataFrame(arr_masked)
df

,0
0,1.0
1,2.0
2,4.0
3,5.0
4,NaN


In [3]:
df2 = df.reindex(['a', 'b', 'c', 'd', 'e', 'f'])
df2.iloc[1, 0] = 2
df2.iloc[4, 0] = 4
df2

,0
a,NaN
b,2.0
c,NaN
d,NaN
e,4.0
f,NaN


In [4]:
df3 = df2.ffill()
df3

,0
a,NaN
b,2.0
c,2.0
d,2.0
e,4.0
f,4.0


In [5]:
obj2 = obj.reindex(['a', 'b', 'c', 'd', 'e'])
obj2

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

Для упорядоченных данных, например временных рядов, иногда желательно произвести интерполяцию, или восполнение,
отсутствующих значений в процессе переиндексации. Это позволяет сделать параметр method; так, если задать для него
значение ffill, то будет произведено прямое восполнение значений:


In [6]:
obj3 = pd.Series(['blue', 'purple', 'yellow'], index=[0, 2, 4])
obj3

0      blue
2    purple
4    yellow
dtype: object

In [7]:
obj3.reindex(range(6), method='ffill')

0      blue
1      blue
2    purple
3    purple
4    yellow
5    yellow
dtype: object

В случае объекта DataFrame функция reindex может изменять строки, столбцы или то и другое. Если ей передать просто
последовательность, то в результирующем объекте переиндексируются строки:

In [8]:
frame = pd.DataFrame(np.arange(9).reshape((3, 3)),
                     index=['a', 'c', 'd'],
                     columns=['Ohio', 'Texas', 'California'])
frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [9]:
frame2 = frame.reindex(['a', 'b', 'c', 'd'])
frame2

,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0


Столбцы можно переиндексировать, задав ключевое слово columns:

In [10]:
states = ['Texas', 'Utah', 'California']
frame.reindex(columns=states)

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


## Задание 1

Аргументы функции reindex: index; method; fill_value; limit; tolerance; level; copy.


In [11]:
# index - указание новых индексов для реиндексирования
print(frame2)
frame.reindex(index=['a', 'b', 'c', 'd', 'e'])

   Ohio  Texas  California
a   0.0    1.0         2.0
b   NaN    NaN         NaN
c   3.0    4.0         5.0
d   6.0    7.0         8.0


,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0
e,NaN,NaN,NaN


In [12]:
# method - метод для заполнения новых значений после реиндексации
print(obj3)
obj3.reindex(index=[0, 2, 4, 5, 6, 7, 8], method='nearest')

0      blue
2    purple
4    yellow
dtype: object


0      blue
2    purple
4    yellow
5    yellow
6    yellow
7    yellow
8    yellow
dtype: object

In [13]:
# fill_value - конкретное значение для заполнения после реиндексации
obj3.reindex(index=[0, 2, 4, 5, 6, 7, 8], fill_value=0)

0      blue
2    purple
4    yellow
5         0
6         0
7         0
8         0
dtype: object

In [14]:
# limit - ограничение на количество автозаполнений с помощью method
obj3.reindex(index=[0, 2, 4, 5, 6, 7, 8], method="ffill", limit=2)

0      blue
2    purple
4    yellow
5    yellow
6    yellow
7       NaN
8       NaN
dtype: object

In [15]:
# tolerance - ограничение на количество автозаполнений с помощью method, но смотрим на разницу индексов (работает с числами и датами)
obj3.reindex(index=[0, 2, 4, 5, 8, 9, 10], method="ffill", tolerance=2)

0       blue
2     purple
4     yellow
5     yellow
8        NaN
9        NaN
10       NaN
dtype: object

In [16]:
# level - указание уровня реиндексации (в случае если у нас несколько уровней индексов)
arrays = [
    ["A", "A", "B", "B"],
    [1, 2, 1, 2]
]
index = pd.MultiIndex.from_arrays(arrays)

s = pd.Series([10, 20, 30, 40], index=index)
s

A  1    10
   2    20
B  1    30
   2    40
dtype: int64

In [17]:
s.reindex(index=["B", "A"], level=0)

B  1    30
   2    40
A  1    10
   2    20
dtype: int64

In [18]:
# copy - указание, делать ли копию при реиндексации
obj3_new = obj3.reindex(copy= True)
obj3[0] = "red"
obj3_new

0      blue
2    purple
4    yellow
dtype: object

In [19]:
obj3_new = obj3.reindex(copy= False)
obj3[0] = "black"
obj3_new

0     black
2    purple
4    yellow
dtype: object

## Удаление элементов из оси

Удалить один или несколько элементов из оси просто, если имеется индексный массив или список, не содержащий этих
значений. Метод drop возвращает новый объект, в котором указанные значения удалены из оси:


In [20]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

obj = pd.Series(np.arange(5.), index=['a', 'b', 'c', 'd', 'e'])
obj

a    0.0
b    1.0
c    2.0
d    3.0
e    4.0
dtype: float64

In [21]:
new_obj = obj.drop('c')
new_obj

a    0.0
b    1.0
d    3.0
e    4.0
dtype: float64

In [22]:
obj.drop(['d', 'c'])

a    0.0
b    1.0
e    4.0
dtype: float64

В случае DataFrame указанные в индексе значения можно удалить из любой оси. Для иллюстрации сначала создадим
объект DataFrame:

In [23]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=['Ohio', 'Colorado', 'Utah', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


Если вызвать drop, указав последовательность меток, то будут удалены строки с такими метками (ось 0):

In [24]:
data.drop(['Colorado', 'Ohio'])

,one,two,three,four
Utah,8,9,10,11
New York,12,13,14,15


Чтобы удалить столбцы, следует передать axis=1 или axis='columns':

In [25]:
data.drop('two', axis=1)

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


In [26]:
data.drop(['two', 'four'], axis='columns')

,one,three
Ohio,0,2
Colorado,4,6
Utah,8,10
New York,12,14


Многие функции, подобные drop, которые модифицируют размер или форму Series или DataFrame, могут изменять объект
на месте, не создавая новый объект:

In [27]:
obj.drop('c', inplace=True)
obj

a    0.0
b    1.0
d    3.0
e    4.0
dtype: float64

inplace удаляет данные совсем

## Доступ по индексу, выборка и фильтрация

Доступ по индексу к объекту Series (obj[...]) работает так же, как для массивов NumPy, с тем отличием, что индексами
могут быть не только целые, но и любые значения из индекса объекта Series. Вот несколько примеров:

In [28]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

obj = pd.Series(np.arange(4.), index=['a', 'b', 'c', 'd'])
obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

## Доступ по индексу, выборка и фильтрация

Доступ по индексу к объекту Series (obj[...]) работает так же, как для массивов NumPy, с тем отличием, что индексами
могут быть не только целые, но и любые значения из индекса объекта Series. Вот несколько примеров:

In [29]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

obj = pd.Series(np.arange(4.), index=['a', 'b', 'c', 'd'])
obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

In [30]:
obj['b']

np.float64(1.0)

In [31]:
obj[1]

/tmp/ipykernel_43393/3621043455.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[1]


np.float64(1.0)

In [32]:
obj[2:4]

c    2.0
d    3.0
dtype: float64

In [33]:
obj[['b', 'a', 'd']]

b    1.0
a    0.0
d    3.0
dtype: float64

In [34]:
obj[[1, 3]]

/tmp/ipykernel_43393/4102577471.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[[1, 3]]


b    1.0
d    3.0
dtype: float64

In [35]:
obj[obj < 2]

a    0.0
b    1.0
dtype: float64

Вырезание с помощью меток отличается от обычного вырезания в Python тем, что конечная точка включается:

In [36]:
obj['b':'c']

b    1.0
c    2.0
dtype: float64

Установка с помощью этих методов модифицирует соответствующий участок Series:

In [37]:
obj['b':'c'] = 5
obj

a    0.0
b    5.0
c    5.0
d    3.0
dtype: float64

Обращение по индексу к DataFrame предназначено для получения одного или нескольких столбцов путем задания одного
значения или последовательности:

In [38]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=['Ohio', 'Colorado', 'Utah', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [39]:
data['two']

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [40]:
data[['three', 'one']]

,three,one
Ohio,2,0
Colorado,6,4
Utah,10,8
New York,14,12


У доступа по индексу есть несколько частных случаев. Во-первых, выборка или вырезание данных с помощью булева
массива:

In [41]:
data[:2]

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7


In [42]:
data[data['three'] > 5]

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


Для удобства предоставляется специальный синтаксис выборки строк data[:2]. Если передать один элемент или список
оператору [], то выбираются столбцы.

Еще одна возможность – доступ по индексу с помощью булева DataFrame, например, порожденного в результате
скалярного сравнения:

In [43]:
data < 5

,one,two,three,four
Ohio,True,True,True,True
Colorado,True,False,False,False
Utah,False,False,False,False
New York,False,False,False,False


In [44]:
data[data < 5] = 0
data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


## Выборка с помощью loc и iloc

Для доступа к строкам DataFrame по индексу с помощью меток я ввел специальные индексные операторы loc и iloc. Они
позволяют выбрать подмножество строк и столбцов DataFrame с применением нотации NumPy, используя либо метки
строк (loc), либо целые числа (iloc).

Выбор одной строки и нескольких столбцов по меткам:


In [45]:
data.loc['Colorado', ['two', 'three']]

two      5
three    6
Name: Colorado, dtype: int64

Произведем аналогичную выборку, но уже по целочисленным индексам:

In [46]:
data.iloc[2, [3, 0, 1]]

four    11
one      8
two      9
Name: Utah, dtype: int64

In [47]:
data.iloc[2]

one       8
two       9
three    10
four     11
Name: Utah, dtype: int64

In [48]:
data.iloc[[1, 2], [3, 0, 1]]

,four,one,two
Colorado,7,0,5
Utah,11,8,9


Обе функции индексирования работают не только с одиночными метками или списками меток, но и со срезами:

In [49]:
data.loc[:'Utah', 'two']

Ohio        0
Colorado    5
Utah        9
Name: two, dtype: int64

In [50]:
data.iloc[:, :3][data.three > 5]

,one,two,three
Colorado,0,5,6
Utah,8,9,10
New York,12,13,14


## Задание 2

Варианты доступа по индексу для объекта DataFrame: df[val]; df.loc[val]; obj.loc[:, val]; df.iloc[val1, val2]; df.iloc[where];
obj.iloc[:, where]; df.loc[where_i, where_j]; df.at[label_i, label_j]; df.iat[label_i, label_j]; метод reindex; методы get_value,
set_value.


In [51]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=['Ohio', 'Colorado', 'Utah', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [52]:
# df[val] - доступ по метке столбца
data["one"]

Ohio         0
Colorado     4
Utah         8
New York    12
Name: one, dtype: int64

In [53]:
data[["one", "three", "four"]]

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


In [54]:
# df.loc[val] - доступ по меткам строк
data.loc["Ohio"]

one      0
two      1
three    2
four     3
Name: Ohio, dtype: int64

In [55]:
data.loc["Ohio", ["two", "three", "four"]]

two      1
three    2
four     3
Name: Ohio, dtype: int64

In [56]:
# obj.loc[:, val] - срез всех строк + столбцы по меткам
data.loc[:, "three"]

Ohio         2
Colorado     6
Utah        10
New York    14
Name: three, dtype: int64

In [57]:
data.loc[:, ['one', 'four']]

,one,four
Ohio,0,3
Colorado,4,7
Utah,8,11
New York,12,15


In [58]:
# df.iloc[val1, val2] - доступ по целочисленным позициям к конкретной ячейке датафрейма
data.iloc[0, 1]

np.int64(1)

In [59]:
# obj.iloc[:, where] — выбор столбцов по позициям
data.iloc[:, 1]

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [60]:
data.loc["Ohio"]

one      0
two      1
three    2
four     3
Name: Ohio, dtype: int64

In [61]:
# df.loc[where_i, where_j] - комбинация меток для столбцов и строк
where_i = ['Ohio', 'Utah']
where_j = ['two', 'three'] 
data.loc[where_i, where_j]

,two,three
Ohio,1,2
Utah,9,10


In [62]:
# df.at[label_i, label_j] - доступ к ячейке датафрейма (строковые данные)
data.at["Ohio", "two"]

np.int64(1)

In [63]:
# df.iat[i, j] - доступ к ячейке датафрейма (числовые значения)
data.iat[0, 1]

np.int64(1)

In [64]:
# reindex - переупорядочивание/расширение осей по указанным меткам
data.reindex(columns=['three', 'one', 'two', 'four', 'five'])

,three,one,two,four,five
Ohio,2,0,1,3,NaN
Colorado,6,4,5,7,NaN
Utah,10,8,9,11,NaN
New York,14,12,13,15,NaN


In [65]:
data.reindex(index=['Ohio', 'Colorado', 'Utah', "Chelyabinsk"], method=None)

,one,two,three,four
Ohio,0.0,1.0,2.0,3.0
Colorado,4.0,5.0,6.0,7.0
Utah,8.0,9.0,10.0,11.0
Chelyabinsk,NaN,NaN,NaN,NaN


In [66]:
# get_value / set_value - получение / назначение значения
data._get_value("Utah", "one")

np.int64(8)

In [67]:
data._set_value("Ohio", "one", 100)
data

,one,two,three,four
Ohio,100,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


ValueError: Length of values (3) does not match length of index (10)